# Spice Hoes — ComfyUI Wan2.1 I2V worker
Google Colab GPU worker using ComfyUI + ComfyUI-GGUF. Q4_K_M is the quality default; switch to Q3_K_M when VRAM is tight. No hosted image/video generation API token is required.

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI || true
%cd /content/ComfyUI
!pip -q install -r requirements.txt
!git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF || true
!pip -q install -r custom_nodes/ComfyUI-GGUF/requirements.txt || true


## Download pinned Wan2.1 I2V models
Default is Q4_K_M (~11.3 GB). Change `WAN_QUANT` to `Q3_K_M` (~8.6 GB) if the assigned GPU is too constrained. Shared text encoder, vision encoder, and VAE are the official ComfyUI-repackaged files.

In [ ]:
from pathlib import Path
import subprocess, os
WAN_QUANT = os.environ.get('WAN_QUANT', 'Q4_K_M')  # Q4_K_M quality default; Q3_K_M low-VRAM fallback
root = Path('/content/ComfyUI/models')
for d in ['unet','text_encoders','clip_vision','vae']:
    (root/d).mkdir(parents=True, exist_ok=True)
files = {
  root/'unet'/f'wan2.1-i2v-14b-480p-{WAN_QUANT}.gguf': f'https://huggingface.co/city96/Wan2.1-I2V-14B-480P-gguf/resolve/main/wan2.1-i2v-14b-480p-{WAN_QUANT}.gguf?download=true',
  root/'text_encoders'/'umt5_xxl_fp8_e4m3fn_scaled.safetensors': 'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
  root/'clip_vision'/'clip_vision_h.safetensors': 'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors',
  root/'vae'/'wan_2.1_vae.safetensors': 'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors',
}
for path, url in files.items():
    if not path.exists() or path.stat().st_size == 0:
        subprocess.run(['wget','-c','--progress=bar:force:noscroll','-O',str(path),url], check=True)
    print(path, round(path.stat().st_size/1024**3, 2), 'GiB')


## Start ComfyUI and verify required nodes
The worker starts in low-VRAM mode. The cell verifies the local server exposes every node used by the Spice API workflow before opening the tunnel.

In [ ]:
import subprocess, time, json, urllib.request, re, os
comfy = subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188','--lowvram'], cwd='/content/ComfyUI')
deadline = time.time() + 120
info = None
while time.time() < deadline:
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/object_info', timeout=5) as r:
            info = json.load(r)
        break
    except Exception:
        time.sleep(2)
if info is None:
    raise RuntimeError('ComfyUI did not become ready')
required = {'UnetLoaderGGUF','CLIPLoader','VAELoader','CLIPVisionLoader','CLIPTextEncode','CLIPVisionEncode','WanImageToVideo','ModelSamplingSD3','KSampler','VAEDecode','CreateVideo','SaveVideo','LoadImage'}
missing = sorted(required - set(info))
if missing:
    raise RuntimeError(f'Missing required ComfyUI nodes: {missing}')
print('ComfyUI preflight OK; required node set is present.')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
tunnel = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
deadline = time.time() + 60
while time.time() < deadline and not url:
    line = tunnel.stdout.readline()
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\\.trycloudflare\\.com', line)
    if m: url = m.group(0)
print('\\nCOMFYUI_BASE_URL=', url)
print('COMFYUI_WORKFLOW=', 'config/comfyui/wan21_i2v_q4_api.json' if WAN_QUANT == 'Q4_K_M' else 'config/comfyui/wan21_i2v_q3_api.json')


Use the printed `COMFYUI_BASE_URL` in the Spice worker environment. Quick Tunnel URLs are temporary and unauthenticated; use them only for attended sessions. For unattended production put an authenticated reverse proxy in front of ComfyUI.